# AI Crypto Hedge Fund MVP

This notebook is the final technical solution for the AI Crypto Hedge Fund assignment. It uses repository modules and committed report artifacts rather than duplicating implementation logic inline.

The notebook is designed to run without network calls. By default it reads committed sample data and committed report artifacts. The authoritative metrics shown in the report artifacts were generated on `beleriand` from the ignored full 1-minute Binance spot bundle documented in `data/manifest.json`.

## 0. Setup, Data Mode, and Helpers

Set `DATA_MODE=sample` for a lightweight smoke run from committed data only. Set `DATA_MODE=full` after preparing or downloading `data/external/binance_spot_1m_120_12mo/`; the notebook still reads committed report artifacts so it stays fast enough for review.

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

from ai_crypto_hedge_fund import project_root
from ai_crypto_hedge_fund.data.loaders import load_price_matrix, snapshot_processed_dir

ROOT = project_root()
DATA_MODE = os.environ.get("DATA_MODE", "sample")
if DATA_MODE not in {"sample", "full"}:
    raise ValueError("DATA_MODE must be 'sample' or 'full'.")

REPORTS = ROOT / "reports"
METRICS = REPORTS / "metrics"
FIGURES = REPORTS / "figures"


def load_json(relative_path: str) -> dict:
    return json.loads((ROOT / relative_path).read_text())


def comparison_frame(payload: dict) -> pd.DataFrame:
    frame = pd.DataFrame(payload["comparison_table"])
    numeric = frame.select_dtypes(include="number").columns
    frame[numeric] = frame[numeric].round(4)
    return frame


def show_figure(relative_path: str, width: int = 900) -> None:
    path = ROOT / relative_path
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        display(Markdown(f"Missing figure: `{relative_path}`"))

ROOT, DATA_MODE

## 1. Data Preparation and Universe Summary

Market data comes from public Binance spot data. The repository includes a compact 1-minute sample for offline smoke runs. The full 120-pair bundle is intentionally ignored by git and documented with checksums for final packaging.

In [ ]:
manifest = load_json("data/manifest.json")
prices = load_price_matrix(snapshot=DATA_MODE)

summary = {
    "data_mode": DATA_MODE,
    "price_rows": len(prices),
    "price_columns": prices.shape[1],
    "first_timestamp": str(prices.index.min()),
    "last_timestamp": str(prices.index.max()),
    "sample_symbols": list(prices.columns[:10]),
    "full_snapshot_symbol_count": manifest["snapshots"]["full"]["symbol_count"],
    "full_processed_bundle": manifest["snapshots"]["full"].get("processed_bundle", {}),
}
summary

## 2.1 Baseline Strategy for a Single Cryptocurrency

The baseline uses BTCUSDT and compares buy-and-hold with a transparent dual moving-average crossover strategy. Signals are shifted by one period before execution and transaction costs are charged from turnover.

In [ ]:
baseline_metrics = load_json("reports/metrics/baseline_metrics.json")
comparison_frame({
    "comparison_table": [
        {"strategy": name, **metrics}
        for name, metrics in baseline_metrics["strategies"].items()
    ]
})[["strategy", "total_return", "annualized_volatility", "sharpe_ratio", "max_drawdown", "turnover"]]

In [ ]:
show_figure("reports/figures/baseline_equity_curve.png")

## 2.2 Econometric, ML, and AI-Agent Strategy for One Asset

The enhanced BTCUSDT experiment compares rolling econometric forecasts, a RandomForest direction classifier, and a deterministic agent layer that combines baseline, econometric, and ML votes with volatility/drawdown controls.

In [ ]:
model_metrics = load_json("reports/metrics/single_asset_model_comparison.json")
comparison_frame(model_metrics)[["strategy", "total_return", "annualized_volatility", "sharpe_ratio", "max_drawdown", "turnover"]]

In [ ]:
model_metrics["training_summary"]

In [ ]:
show_figure("reports/figures/single_asset_model_comparison.png")

## 2.3 Static Portfolio Management for 5-7 Coins

The static portfolio expands from BTCUSDT to six liquid USDT pairs: BTC, ETH, BNB, SOL, XRP, and ADA. Weights are fitted on the training period only and evaluated on the out-of-sample test period.

In [ ]:
static_metrics = load_json("reports/metrics/static_portfolio_metrics.json")
comparison_frame(static_metrics)[["strategy", "total_return", "annualized_volatility", "sharpe_ratio", "max_drawdown", "effective_assets", "max_weight"]]

In [ ]:
{
    "selected_method": static_metrics["selected_method"],
    "selection_criterion": static_metrics["selection_criterion"],
    "weights": static_metrics["weights"][static_metrics["selected_method"]],
}

In [ ]:
show_figure("reports/figures/static_portfolio_equity_curve.png")
show_figure("reports/figures/static_portfolio_weights.png")

## 2.4 Dynamic Portfolio Rebalancing

The dynamic experiment compares the selected static max-Sharpe portfolio with weekly inverse-volatility rebalancing and threshold rebalancing when passive holding drift exceeds 2 percentage points.

In [ ]:
rebalancing_metrics = load_json("reports/metrics/rebalancing_metrics.json")
comparison_frame(rebalancing_metrics)[["strategy", "total_return", "annualized_volatility", "sharpe_ratio", "max_drawdown", "turnover", "rebalance_events"]]

In [ ]:
{
    "selected_strategy": rebalancing_metrics["selected_strategy"],
    "event_summary": rebalancing_metrics["event_summary"],
}

In [ ]:
show_figure("reports/figures/rebalancing_comparison.png")

## 2.5 Large-Universe Portfolio with Dynamic Rebalancing

The large-universe experiment uses the full 120-pair Binance spot snapshot and compares a broad equal-weight reference with two sparse weekly dynamic allocators. The sparse allocators rank assets from trailing momentum and volatility, select the top 20 active symbols, cap each asset at 8%, and scale gross exposure down when trailing portfolio volatility is above target.

In [ ]:
large_universe_metrics = load_json("reports/metrics/large_universe_metrics.json")
comparison_frame(large_universe_metrics)[["strategy", "total_return", "annualized_volatility", "sharpe_ratio", "max_drawdown", "turnover", "rebalance_events", "average_active_assets"]]

In [ ]:
selected_assets = pd.read_csv(ROOT / "reports" / "metrics" / "large_universe_selected_assets.csv")
{
    "universe_size": large_universe_metrics["universe_size"],
    "selected_strategy": large_universe_metrics["selected_strategy"],
    "data_files": large_universe_metrics["data_files"],
    "selected_assets_preview": selected_assets.head(10).to_dict(orient="records"),
}

In [ ]:
show_figure("reports/figures/large_universe_equity_curve.png")

## 3. Summary, Limitations, and Next Steps

Key results:

- The BTCUSDT RandomForest policy reduced drawdown versus buy-and-hold during the single-asset test period, but the single-asset strategies remained challenged by costs and noisy 1-minute labels.
- The six-asset static max-Sharpe portfolio was the best small-universe strategy by out-of-sample Sharpe.
- Simple dynamic inverse-volatility rebalancing did not improve the six-asset portfolio after turnover.
- In the 120-pair experiment, broad equal weight outperformed the sparse weekly momentum strategies on the selected out-of-sample window.

Limitations:

- All trading is simulated; there is no live execution.
- Fills use minute-close prices and simplified transaction costs.
- The full data bundle is ignored by git and will be published separately in the final data-publication task.
- The ML and momentum signals are deliberately simple MVP signals, not production alpha research.

Production next steps:

- Add exchange-specific slippage, fee tiers, and order-size constraints.
- Add monitoring for stale data, failed orders, exposure drift, realized costs, drawdown, and active-set instability.
- Add fail-safes: stale-data block, maximum rebalance turnover, drawdown stop, exchange outage mode, and manual kill switch.
- Re-test on multiple time periods before treating any strategy as deployable.